# Financial Data with AKShare

AKShare is a Python-based financial data interface library that can be used to collect, clean, and analyze Chinese market data for academic or teaching purposes.

**Note:** This notebook is China-market specific. Because online financial-data APIs can change over time, some AKShare endpoints may occasionally change or become temporarily unavailable.

In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "data").exists():
            return candidate
    return current

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
ASSETS_DIR = PROJECT_ROOT / "assets"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT


In [ ]:
MODULE_OUTPUT_DIR = OUTPUT_DIR / "06_financial_data"
MODULE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODULE_OUTPUT_DIR

**Optional setup**

In [ ]:
%pip install -q akshare ipywidgets seaborn scipy

In [ ]:
import akshare as ak
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Part 1. Obtain stock data and compute basic statistics

# Q1. How can we check the stock codes and names of listed companies on the A-share market?

For `stock_info_sh_name_code`, the `symbol` parameter uses Chinese board names such as:
- `主板A股` = Main Board A-shares
- `主板B股` = Main Board B-shares
- `科创板` = STAR Market

In [ ]:
board_symbol = "主板A股"
stock_info = ak.stock_info_sh_name_code(symbol=board_symbol)
stock_info.head()

In [ ]:
# Daily snapshot of the A-share market.
stock_today = ak.stock_zh_a_spot_em()
stock_today.head()

# Q2. How can we obtain stock data?

## Q2.1. Single-stock example: Shanghai Pudong Development Bank (`600000`)

In [ ]:
stock_pufa = ak.stock_zh_a_hist(
    symbol="600000",
    period="daily",
    start_date="20240101",
    end_date="20250101",
    adjust="hfq",  # backward-adjusted prices
)

stock_pufa = stock_pufa.rename(columns={
    "日期": "date",
    "开盘": "open",
    "收盘": "close",
    "涨跌幅": "pct_change_reported",
})

stock_pufa["date"] = pd.to_datetime(stock_pufa["date"])
stock_pufa["daily_return"] = stock_pufa["close"].pct_change()

stock_pufa.head()

## Q2.2. Multi-stock example

In [ ]:
symbols = ["600000", "600004", "600006", "600007"]

stock_name_map = {
    "600000": "Shanghai Pudong Development Bank",
    "600004": "Baiyun Airport",
    "600006": "Dongfeng Motor",
    "600007": "China World Trade Center",
}

dfs = []
for symbol in symbols:
    data = ak.stock_zh_a_hist(
        symbol=symbol,
        period="daily",
        start_date="20240101",
        end_date="20241231",
        adjust="hfq",
    ).rename(columns={
        "日期": "date",
        "收盘": "close",
        "开盘": "open",
    })

    data["stock_code"] = symbol
    data["stock_name"] = stock_name_map[symbol]
    data["date"] = pd.to_datetime(data["date"])
    data["daily_return"] = data["close"].pct_change()
    dfs.append(data)

stock_df = pd.concat(dfs, ignore_index=True)
stock_df.head()

# Q3. How can we calculate annualized return, annualized volatility, and the Sharpe ratio?

Assume 242 trading days per year.

- Annualized return = mean daily return × 242
- Annualized volatility = standard deviation of daily return × √242
- Sharpe ratio = (annualized return − risk-free rate) / annualized volatility

This notebook uses an annualized risk-free rate of 0.015 (1.5%).

In [ ]:
stock_pufa[["date", "close", "daily_return"]].head()

In [ ]:
pufa_return = stock_pufa["daily_return"].mean() * 242
pufa_return

In [ ]:
pufa_vol = stock_pufa["daily_return"].std() * np.sqrt(242)
pufa_vol

In [ ]:
pufa_sharpe = (pufa_return - 0.015) / pufa_vol
pufa_sharpe

# Q4. Grouped descriptive statistics

In [ ]:
summary_table = stock_df.groupby("stock_name")[["daily_return", "close"]].describe().round(4).T
summary_table

# Part 2. Visualize stock data

# Q5. How can we create basic visualizations for stock data?

## Q5.1. Opening and closing prices for Shanghai Pudong Development Bank

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(stock_pufa["date"], stock_pufa["open"], linestyle="-.", linewidth=1, marker=".", color="#0f0f0f80")
plt.plot(stock_pufa["date"], stock_pufa["close"], color="yellow")
plt.xlabel("Date")
plt.ylabel("Price")
plt.title("Shanghai Pudong Development Bank: Open vs. Close")
plt.legend(["Opening price", "Closing price"], loc="best")
plt.savefig(MODULE_OUTPUT_DIR / "open_close_lineplot.png", bbox_inches="tight")

In [ ]:
%pip install -q seaborn

## Q5.2. Multi-stock line plot

In [ ]:
import seaborn as sns

sns.lineplot(data=stock_df, x="date", y="daily_return", hue="stock_name")
plt.savefig(MODULE_OUTPUT_DIR / "multi_stock_daily_return_lineplot.png", bbox_inches="tight")

## Q5.3. Histograms for multiple stocks

In [ ]:
g = sns.FacetGrid(stock_df, col="stock_name", col_wrap=2)
g.map(sns.histplot, "daily_return", color="pink", bins=20)

## Q5.4. Cumulative return plot

In [ ]:
stock_df = stock_df.sort_values(["stock_name", "date"]).copy()
stock_df["cumulative_return"] = stock_df.groupby("stock_name")["daily_return"].cumsum()

sns.lineplot(data=stock_df, x="date", y="cumulative_return", hue="stock_name")
plt.savefig(MODULE_OUTPUT_DIR / "cumulative_return_lineplot.png", bbox_inches="tight")

# Part 3. Markowitz mean-variance portfolio model

This section uses Monte Carlo simulation and constrained optimization to explore random portfolios, the maximum-Sharpe portfolio, the minimum-variance portfolio, and the efficient frontier.

In [ ]:
from scipy.optimize import minimize
import scipy.optimize as sco

returns = (
    stock_df[["stock_code", "daily_return", "date"]]
    .pivot_table(values="daily_return", index="date", columns="stock_code")
    .dropna(how="all")
)
returns.head()

# Q6. How can we generate many random portfolios with Monte Carlo simulation?

In [ ]:
noa = 4
risk_free = 0.015

port_returns = []
port_variance = []
sharpe_ratios = []

for _ in range(4000):
    weights = np.random.random(noa)
    weights /= np.sum(weights)

    port_return = np.sum(returns.mean() * 242 * weights)

    if port_return > 0:
        port_returns.append(port_return)
        port_variance.append(np.sqrt(np.dot(weights.T, np.dot(returns.cov() * 242, weights))))
        sharpe_ratio = (port_return - risk_free) / port_variance[-1]
        sharpe_ratios.append(sharpe_ratio)

port_returns = np.array(port_returns)
port_variance = np.array(port_variance)
sharpe_ratios = np.array(sharpe_ratios)

In [ ]:
plt.figure(figsize=(8, 3))
plt.scatter(port_variance, port_returns, c=(port_returns - risk_free) / port_variance, marker="o")
plt.xlabel("Expected volatility")
plt.ylabel("Expected return")
plt.grid(True)
plt.colorbar(label="Sharpe ratio")
plt.savefig(MODULE_OUTPUT_DIR / "monte_carlo_portfolios.png", bbox_inches="tight")

# Q7. How can we find the portfolio with the largest Sharpe ratio?

In [ ]:
max_sharpe_idx = np.argmax(sharpe_ratios)
max_sharpe_return = port_returns[max_sharpe_idx]
max_sharpe_variance = port_variance[max_sharpe_idx]
max_sharpe_ratio = sharpe_ratios[max_sharpe_idx]

print(f"Largest Sharpe ratio: {max_sharpe_ratio:.4f}")
print(f"Expected return: {max_sharpe_return:.4f}")
print(f"Expected volatility: {max_sharpe_variance:.4f}")

In [ ]:
def statistics(weights):
    weights = np.array(weights)
    port_return = np.sum(returns.mean() * 242 * weights)
    port_variance = np.sqrt(np.dot(weights.T, np.dot(returns.cov() * 242, weights)))
    return np.array([port_return, port_variance, port_return / port_variance])

def min_sharpe(weights):
    return -statistics(weights)[2]

bnds = ((0, 1),) * noa
cons = {"type": "eq", "fun": lambda x: np.sum(x) - 1}

opts = sco.minimize(min_sharpe, noa * [1.0 / noa], method="SLSQP", bounds=bnds, constraints=cons)
opts

# Q8. How can we find the portfolio that minimizes variance?

In [ ]:
def min_variance(weights):
    return statistics(weights)[1]

optv = sco.minimize(min_variance, noa * [1.0 / noa], method="SLSQP", bounds=bnds, constraints=cons)

print(f"Optimal portfolio weights: {optv.x}")
print(f"Minimum variance: {optv.fun:.4f}")

# Q9. How can we draw the efficient frontier?

In [ ]:
target_returns = np.linspace(returns.mean().mean() * 242, returns.mean().max() * 242, 100)

target_variance = []

for tar in target_returns:
    cons = (
        {"type": "eq", "fun": lambda x: statistics(x)[0] - tar},
        {"type": "eq", "fun": lambda x: np.sum(x) - 1},
    )

    res = sco.minimize(min_variance, noa * [1.0 / noa], method="SLSQP", bounds=bnds, constraints=cons)
    target_variance.append(res["fun"])

target_variance = np.array(target_variance)

In [ ]:
plt.figure(figsize=(8, 3))
plt.scatter(port_variance, port_returns, c=(port_returns - risk_free) / port_variance, marker="o")
plt.scatter(target_variance, target_returns, c=target_returns / target_variance, cmap="autumn", marker="x", s=5.0)
plt.plot(statistics(opts["x"])[1], statistics(opts["x"])[0], "r*", markersize=15.0)
plt.plot(statistics(optv["x"])[1], statistics(optv["x"])[0], "y*", markersize=15.0)
plt.xlabel("Expected volatility")
plt.ylabel("Expected return")
plt.grid(True)
plt.colorbar(label="Sharpe ratio")
plt.savefig(MODULE_OUTPUT_DIR / "efficient_frontier.png", bbox_inches="tight")